In [4]:
!pip install requests -quiet


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -u


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings('ignore')
print("All libraries imported successfully")
print("Pandas version:",(pd.__version__))
print("Numpy version:",(np.__version__))
print("Requests version:",(requests.__version__))

All libraries imported successfully
Pandas version: 2.2.2
Numpy version: 2.0.2
Requests version: 2.32.4


#ETL-Extracting Transforming Loading

In [6]:
import pandas as pd
df=pd.read_csv('messy_sales_data.csv')
print("number of rows and columns",df.shape)
print("number of rows ",df.shape[0])
print("number of  columns",df.shape[1])

number of rows and columns (30, 9)
number of rows  30
number of  columns 9


In [7]:
df.columns.tolist()

['order_id',
 'customer_name',
 'product',
 'category',
 'quantity',
 'unit_price',
 'order_date',
 'city',
 'sales_rep']

In [8]:
df.head(5)

,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [9]:
print('='*30)
print("DATA QUALITY BUSINESS REPORT")
print('='*30)

print("Missing values per column")
print(df.isnull().sum())

print(f'\n[1] Duplicate Rows: {df.duplicated().sum()}')
print(f'\n[2] Data Types: \n {df.dtypes}')
print(f'\n [3]Unique Categories:{df['category'].unique()}')
print(f'\n[4] Sample customer names:{df['customer_name'].dropna()[:8]}')
print(f'\n[5] Sample order data values: {df['order_date'].unique()[:6]}')


DATA QUALITY BUSINESS REPORT
Missing values per column
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[1] Duplicate Rows: 0

[2] Data Types: 
 order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

 [3]Unique Categories:['Electronics' 'Accessories' nan]

[4] Sample customer names:0    Ramesh Kumar
1      Priya Nair
2      AMIT VERMA
3    Sunita Patel
4    Ramesh Kumar
5     kiran mehta
6    Deepak Singh
8      Ananya Das
Name: customer_name, dtype: object

[5] Sample order data values: ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12']


DATA CENTER->MEAN,MEDIUN,MODE

In [10]:
copy_df=df.copy()
print("Working copy created: ",df.shape)



Working copy created:  (30, 9)


In [11]:
print(df.isnull().sum())
print("before filling nulls: ",df.isnull().sum().sum(),'total missing values')

df['customer_name'].fillna('Unknown Customer',inplace=True)
median_qty=df['quantity'].median()
df['quantity'].fillna(median_qty,inplace=True)
print("filling missing quantity with median",median_qty)
df['category'].fillna('Uncatogarized',inplace=True)
print("After fixing nulls: ",df.isnull().sum().sum(),'total missing values')


order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64
before filling nulls:  7 total missing values
filling missing quantity with median 2.0
After fixing nulls:  1 total missing values


In [12]:
print(f'before deduplication: {len(df)} rows')
print(f'Duplicate rows:{df.duplicated().sum()}')

print("Duplicate rows")
print(df[df.duplicated(keep=False)])

df.drop_duplicates(inplace=True)
print(f'after deduplication: {len(df)} rows')
print(f'rows removed:{len(df)-len(copy_df)}')


before deduplication: 30 rows
Duplicate rows:0
Duplicate rows
Empty DataFrame
Columns: [order_id, customer_name, product, category, quantity, unit_price, order_date, city, sales_rep]
Index: []
after deduplication: 30 rows
rows removed:0


In [13]:
print('sample date before parsing')
print(df['order_date'].head(8).tolist())

df['order_date']=pd.to_datetime(
    df['order_date'],
    dayfirst=False,
    errors='coerce'
)
nat_count=df['order_date'].isnull().sum()
print(f'\n unpredictable dates(NAT): {nat_count}')

df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')
df['year']=df['order_date'].dt.year

print('\n Sample dates after parsing')
print(df[['order_date','year','month','month_name']].head(8))



sample date before parsing
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']

 unpredictable dates(NAT): 2

 Sample dates after parsing
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January
5        NaT     NaN    NaN        NaN
6 2024-01-12  2024.0    1.0    January
7 2024-01-13  2024.0    1.0    January


In [14]:
print('before standardization: ',df['customer_name'].unique()[:6])
df['customer_name']=(
    df['customer_name']
    .str.strip()
    .str.title()
)

print("After Standardization: ",df['customer_name'].unique()[:6])
print("keyboard rows with electronics category: ")
wrong_mask=(df['product']=='keyboard')&(df['category']=='electronics')
print(df[wrong_mask][['product','category']])
df.loc[wrong_mask,'category']='accessories'
print("after fix: unique categories",df['category'].unique())


before standardization:  ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
After Standardization:  ['Ramesh Kumar' 'Priya Nair' 'Amit Verma' 'Sunita Patel' 'Kiran Mehta'
 'Deepak Singh']
keyboard rows with electronics category: 
Empty DataFrame
Columns: [product, category]
Index: []
after fix: unique categories ['Electronics' 'Accessories' 'Uncatogarized']


In [15]:
df['quantity']=pd.to_numeric(df['quantity'],errors='coerce')
df['unit_price']=pd.to_numeric(df['unit_price'],errors='coerce')

df['revenue']=df['quantity']*df['unit_price']
print("revenue column created: ")
print(df[['customer_name','product','quantity','unit_price','revenue']].head(5))
print("total revenue across all orders: ", f"{df['revenue'].sum():,.0f}")

revenue column created: 
  customer_name   product  quantity  unit_price  revenue
0  Ramesh Kumar    Laptop       2.0       45000  90000.0
1    Priya Nair       NaN       1.0       15000  15000.0
2    Amit Verma  Keyboard       3.0        1200   3600.0
3  Sunita Patel   Monitor       2.0       22000  44000.0
4  Ramesh Kumar    Laptop       2.0       45000  90000.0
total revenue across all orders:  818,000


In [16]:
print("="*55)
print('post-cleaning validation report')
print("="*55)
print("original rows: ",len(df))
print("cleaned rows: ",len(copy_df))
print("rows removed: ",len(df)-len(copy_df),"duplicates")
print("missing values: ",df.isnull().sum().sum())
print("duplicates: ",df.duplicated().sum())
print("date nulls: ",df['order_date'].isnull().sum())
print("revenue null: ",df['revenue'].isnull().sum())
print("caategories: ",sorted(df['category'].unique()))
print("="*55)

all_clean=(
    df.isnull().sum().sum()==0 and
    df.duplicated().sum()==0
)
print("data is clean: ",all_clean)

post-cleaning validation report
original rows:  30
cleaned rows:  30
rows removed:  0 duplicates
missing values:  9
duplicates:  0
date nulls:  2
revenue null:  0
caategories:  ['Accessories', 'Electronics', 'Uncatogarized']
data is clean:  False


In [17]:
product_rev=(
    df
    .groupby('product')
    ['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue',ascending=False)
)
print("="*55)
print("revenue by product: ")
print("="*55)
print(product_rev.to_string(index=False))
category_summary=df.groupby('category').agg(
    total_revenue=('revenue','sum'),
    avg_order_value=('revenue','mean'),
    num_orders=('order_id','count'),
    unique_products=('product','nunique')
).round(2).reset_index()

print("="*55)
print("category summary: ")
print("="*55)
print(category_summary.to_string(index=False))


revenue by product: 
   product  revenue
    Laptop 540000.0
   Monitor 154000.0
Headphones  28000.0
     Mouse  20800.0
  Keyboard  20400.0
    Webcam  20000.0
   USB Hub  19800.0
category summary: 
     category  total_revenue  avg_order_value  num_orders  unique_products
  Accessories        76200.0          5861.54          13                4
  Electronics       697800.0         43612.50          16                4
Uncatogarized        44000.0         44000.00           1                1


In [18]:
df.to_csv('clean_sales_data.csv',index=False)

print('cleaned data saved to: clean_sales_data.csv')
print(f'final dataset:{df.shape[0]}rows x{df.shape[1]} columns')
print('\nETL Pipeline for Sales Data: COMPLETE')
print(' EXTRACT -> messy_sales_data.csv loaded')
print(' TRANSFORM -> messy_sales_data.csv cleaned')
print(' LOAD -> clean_sales_data.csv saved')

cleaned data saved to: clean_sales_data.csv
final dataset:30rows x13 columns

ETL Pipeline for Sales Data: COMPLETE
 EXTRACT -> messy_sales_data.csv loaded
 TRANSFORM -> messy_sales_data.csv cleaned
 LOAD -> clean_sales_data.csv saved


In [19]:
API_KEY='847a847a1ed220f0b9c47791'
BASE_URL='https://api.openweathermap.org/data/2.5/weather'

CITIES=['Mumbai','Delhi','Bangalore','Chennai','Hydrabad','Kolkata','Pune','Jaipur']
print("api configured for", len(CITIES),"cities")
print("Cities: ",CITIES)

api configured for 8 cities
Cities:  ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hydrabad', 'Kolkata', 'Pune', 'Jaipur']


#Practice Questions

In [20]:
import pandas as pd

df = pd.read_csv('messy_sales_data.csv') #extract
print(df.head())

   order_id customer_name   product     category  quantity  unit_price  \
0      1001  Ramesh Kumar    Laptop  Electronics       2.0       45000   
1      1002    Priya Nair       NaN  Electronics       1.0       15000   
2      1003    AMIT VERMA  Keyboard  Accessories       3.0        1200   
3      1004  Sunita Patel   Monitor  Electronics       NaN       22000   
4      1005  Ramesh Kumar    Laptop  Electronics       2.0       45000   

   order_date       city    sales_rep  
0  2024-01-05     Mumbai  Anil Sharma  
1  2024-01-07      Delhi   Sunita Rao  
2  2024-01-08  Bangalore  Anil Sharma  
3  2024-01-10    Chennai   Ravi Kumar  
4  2024-01-05     Mumbai  Anil Sharma  


In [22]:
#transform
df = df.drop_duplicates()
df = df.dropna()
df['revenue'] = df['quantity'] * df['unit_price']

In [23]:
#Load
df.to_csv('clean_sales_data.csv', index=False)

In [24]:
df = df.drop_duplicates(subset=['customer_name', 'product'])

print(df.head())

   order_id customer_name     product     category  quantity  unit_price  \
0      1001  Ramesh Kumar      Laptop  Electronics       2.0       45000   
2      1003    AMIT VERMA    Keyboard  Accessories       3.0        1200   
5      1006   kiran mehta       Mouse  Accessories      10.0         800   
6      1007  Deepak Singh  Headphones  Electronics       2.0        3500   
8      1009    Ananya Das      Laptop  Electronics       1.0       45000   

   order_date       city    sales_rep  revenue  
0  2024-01-05     Mumbai  Anil Sharma  90000.0  
2  2024-01-08  Bangalore  Anil Sharma   3600.0  
5  07-01-2024       Pune   Sunita Rao   8000.0  
6  2024-01-12  Hyderabad   Ravi Kumar   7000.0  
8  2024-01-15    Kolkata   Sunita Rao  45000.0  


In [27]:
#fillna(0)
df['sales'] = df['sales_rep'].fillna(0)

In [29]:
df['price'] = df['unit_price'].fillna(df['unit_price'].median())